# Kiko & Friends - GPU Render Worker
This script runs continuously on a Colab T4/A100 GPU instance. It mounts Google Drive, polls the `Jarvis_Render_Queue` for new scripts, generates 3D video clips using a lightweight Diffusers pipeline, and drops the finished MP4s into `Jarvis_Completed_Renders`.

In [ ]:
%pip install -U diffusers transformers accelerate imageio[ffmpeg]

from google.colab import drive
import os
import time
import json
import torch
from diffusers import CogVideoXImageToVideoPipeline
from diffusers.utils import export_to_video
from PIL import Image

# Mount Google Drive
drive.mount('/content/drive')

QUEUE_DIR = '/content/drive/MyDrive/Jarvis_Render_Queue'
COMPLETED_DIR = '/content/drive/MyDrive/Jarvis_Completed_Renders'

os.makedirs(QUEUE_DIR, exist_ok=True)
os.makedirs(COMPLETED_DIR, exist_ok=True)

print("Directories verified.")

In [ ]:
# Load Pipeline
print("Loading CogVideoX-I2V Pipeline...")
pipeline = CogVideoXImageToVideoPipeline.from_pretrained(
    "THUDM/CogVideoX-5b-I2V", 
    torch_dtype=torch.float16
)
# Strict VRAM optimization for Free Tier T4 GPUs (15GB)
pipeline.enable_sequential_cpu_offload()
pipeline.vae.enable_slicing()
print("Pipeline loaded successfully with VRAM optimizations.")

In [ ]:
def process_queue():
    while True:
        files = [f for f in os.listdir(QUEUE_DIR) if f.endswith('.json')]
        if not files:
            print("Queue empty. Waiting 30s...")
            time.sleep(30)
            continue
        
        script_file = files[0]
        script_path = os.path.join(QUEUE_DIR, script_file)
        
        print(f"Found script: {script_file}. Processing...")
        with open(script_path, 'r') as f:
            scenes = json.load(f)
            
        # For each scene in script
        for scene in scenes:
            scene_idx = str(scene.get('scene_number', 0)).zfill(3)
            print(f"Rendering scene {scene_idx}...")
            
            # Extract text prompt
            scene_prompt = scene.get('visual_prompt', '')
            
            # Load the uploaded frame for this scene
            frame_file = f"scene_{scene_idx}.png"
            frame_path = os.path.join(QUEUE_DIR, frame_file)
            if os.path.exists(frame_path):
                base_img = Image.open(frame_path).convert('RGB')
                os.remove(frame_path)
            else:
                print(f"Warning: {frame_file} not found. Falling back to solid color.")
                base_img = Image.new('RGB', (1024, 576), color = 'lightblue')
            
            # Generate Video (CogVideoX uses prompt and image)
            generator = torch.manual_seed(42)
            print(f"Generating video for prompt: '{scene_prompt}'")
            frames = pipeline(prompt=scene_prompt, image=base_img, num_inference_steps=50, guidance_scale=6.0, generator=generator).frames[0]
            
            # Export MP4
            out_file = f"scene_{scene_idx}.mp4"
            out_path = os.path.join(COMPLETED_DIR, out_file)
            export_to_video(frames, out_path, fps=8)
            print(f"Saved {out_file} to Completed Renders.")
            
        # Delete script to signify completion
        print("Removing script from queue.")
        os.remove(script_path)

# Start Worker loop
process_queue()